## Syntax differences


|  Opération                                 |  `pandas`                           |  `polars`                                                   |
| -------------------------------------------- | ------------------------------------- | ------------------------------------------------------------- |
|  Créer un DataFrame                         | `pd.DataFrame({...})`                 | `pl.DataFrame({...})`                                         |
|  Voir les premières lignes                 | `df.head()`                           | `df.head()`                                                   |
|  Accéder à une colonne                     | `df["col"]` ou `df.col`               | `df["col"]` ou `df.select("col")` ou `df["col"]`              |
|  Créer une nouvelle colonne                | `df["z"] = df["x"] + df["y"]`         | `df.with_columns((pl.col("x") + pl.col("y")).alias("z"))`     |
|  Supprimer une colonne                      | `df.drop(columns=["x"])`              | `df.drop("x")`                                                |
|  Supprimer les colonnes d'un type          | `df = df.drop(columns=df.select_dtypes(include='object').columns)` | `df.drop(pl.selectors.bool())` | 
|  Sélection de colonnes                     | `df[["a", "b"]]`                      | `df.select(["a", "b"])`                                       |
|  Sélection par pattern                     | `df.filter(like="foo")`               | `df.select(pl.col("^foo.*$"))` (regex)                        |
|  Filtrage conditionnel                     | `df[df["x"] > 3]`                     | `df.filter(pl.col("x") > 3)`                                  |
|  Filtrage avec plusieurs conditions        | `df[(df["x"] > 3) & (df["y"] < 1)]`   | `df.filter((pl.col("x") > 3) & (pl.col("y") < 1))`            |
|  Colonnes numériques uniquement            | `df.select_dtypes(include=np.number)` | `df.select(pl.selectors.numeric())`                        |
|  Renommer les colonnes                    | `df.rename(columns={"a": "b"})`       | `df.rename({"a": "b"})`                                       |
|  Convertir en datetime                     | `pd.to_datetime(df["ts"])`            | `df_with_columns(pl.col("ts").str.strptime(pl.Datetime)`                      |
|  Extraire l’année/mois/jour d’une datetime | `df["ts"].dt.year`                    | `df_with_columns(pl.col("ts").dt.year())`                                      |
|  Groupby + agg                             | `df.groupby("a").agg({"b": "mean"})`  | `df.groupby("a").agg(pl.col("b").mean())`                     |
|  Rolling mean                              | `df["x"].rolling(window=3).mean()`    | `df_with_columns(pl.col("x").rolling_mean(3))`                                 |
|  Trier                                     | `df.sort_values("x")`                 | `df.sort("x")`                                                |
|  Valeurs uniques                           | `df["x"].unique()`                    | `df.select("x").unique()`                                     |
|  Valeurs nulles                             | `df.isnull()`                         | `df.select(pl.all().is_null())`                               |
|  Drop NA                                    | `df.dropna()`                         | `df.drop_nulls()`                                             |
|  Cumulative sum                            | `df["x"].cumsum()`                    | `df.select(pl.col("x").cumsum())`                             |
|  Stack / Melt                              | `df.melt()`                           | `df.melt(id_vars=..., value_vars=...)`                        |
|  Pipeline (chained ops)                    | pas naturel                           | `.with_columns(...).filter(...).select(...)`                  |
|  Lazy DataFrame                            | (non dispo)                         | `df.lazy()`                                                   |


## Code Snippets

### Polars

In [12]:
from pathlib import Path
import polars as pl

output_dir = Path("benchmark_io")
csv_path = output_dir / "data.csv"
parquet_path = output_dir / "data.parquet"

df = (
    pl.read_parquet(parquet_path)
    .with_columns([
        (pl.col("value")**2).alias("squared"),
        pl.col("timestamp").dt.hour().alias("hour")
    ])
    .filter(pl.col("squared") > 0)
    .group_by("hour")
    .agg(pl.col("squared").mean().alias("avg_squared"))
    .sort("hour")
)

df


hour,avg_squared
i8,f64
0,1.004174
1,0.998984
2,1.003375
3,0.996249
4,1.000523
…,…
19,0.996688
20,0.999413
21,0.997973


### Pandas

In [13]:
from pathlib import Path
import pandas as pd

output_dir = Path("benchmark_io")
csv_path = output_dir / "data.csv"
parquet_path = output_dir / "data.parquet"

df = pd.read_parquet(parquet_path)
df["squared"] = df["value"] ** 2
df["hour"] = df["timestamp"].dt.hour

df_filtered = df[df["squared"] > 0]

df_grouped = df_filtered.groupby("hour", as_index=False)["squared"].mean()
df_grouped.rename(columns={"squared": "avg_squared"}, inplace=True)


df_grouped = df_grouped.sort_values("hour").reset_index(drop=True)

df_grouped

,hour,avg_squared
0,0,1.004174
1,1,0.998984
2,2,1.003375
3,3,0.996249
4,4,1.000523
5,5,0.993732
6,6,0.998207
7,7,1.004607
8,8,0.994749
9,9,1.001897


## Benchmarks

In [11]:
import pandas as pd
from datetime import datetime
import polars as pl
import numpy as np
import time
from pathlib import Path

# --- FONCTION UTILITAIRE DE BENCHMARK ---
def benchmark(name,func):
    start = time.time()
    func()
    return time.time() - start

def print_results(title, results):
    print(f"\n=== {title} ===")
    print(f"{'Operation':35} | {'pandas':>10} | {'polars':>10} | {'times faster (Polars)':>17}")
    print("-" * 80)
    for k, (pandas_time, polars_time) in results.items():
        if pandas_time == 0:
            pct = float('nan')
        else:
            pct = pandas_time / polars_time
        print(f"{k:35} | {pandas_time:10.4f} | {polars_time:10.4f} | {pct:.2f}")


# --- DATASET ---
n = 1_000_000
date_range = pd.date_range("2024-01-01", periods=n, freq="min")
categories = ["a", "b", "c"]

df_pd = pd.DataFrame({
    "timestamp":  np.tile(date_range, len(categories)),
    "category": np.repeat(categories, len(date_range)),
    "value": np.random.randn(n * len(categories)),
})

df_pl = pl.from_pandas(df_pd)

# --- OPÉRATIONS SUR TIME SERIES ---
ts_results = {}

ts_results["Filter (category == a)"] = (
    benchmark("pandas", lambda: df_pd[df_pd["category"] == "a"]),
    benchmark("polars", lambda: df_pl.filter(pl.col("category") == 'a'))
)

start_date = datetime(2024, 2, 1)
end_date = datetime(2024, 2, 28, 23, 59, 59)


ts_results["Filter (timestamp between)"] = (
    benchmark("pandas", lambda: df_pd[(df_pd["timestamp"] >= start_date) & (df_pd["timestamp"] <= end_date)]),
    benchmark("polars", lambda: df_pl.filter((pl.col('timestamp').is_between(start_date, end_date))))
)

ts_results["Mean per day"] = (
    benchmark("pandas", lambda: df_pd.set_index(["timestamp"]).resample("D")["value"].mean()),
    benchmark("polars", lambda: df_pl.sort('timestamp', 'category').group_by_dynamic("timestamp", every="1d", period="1d").agg([
        pl.col("value").mean().alias("mean_value")
    ]))
)


ts_results["Extract time features"] = (
    benchmark("pandas", lambda: df_pd["timestamp"].dt.hour + df_pd["timestamp"].dt.dayofweek),
    benchmark("polars", lambda: df_pl.with_columns([
        pl.col("timestamp").dt.hour().alias('hour'),
        pl.col("timestamp").dt.weekday().alias('weekday')
    ]))
)



# --- I/O CSV & PARQUET ---
io_results = {}
output_dir = Path("benchmark_io")
output_dir.mkdir(exist_ok=True)
csv_path = output_dir / "data.csv"
parquet_path = output_dir / "data.parquet"


# CSV
io_results["CSV write"] = (
    benchmark("pandas", lambda: df_pd.to_csv(csv_path, index=False)),
    benchmark("polars", lambda: df_pl.write_csv(csv_path))
)

io_results["CSV read"] = (
    benchmark("pandas", lambda: pd.read_csv(csv_path, parse_dates=["timestamp"])),
    benchmark("polars", lambda: pl.read_csv(csv_path, try_parse_dates=True))
)

# Parquet
io_results["Parquet write"] = (
    benchmark("pandas", lambda: df_pd.to_parquet(parquet_path, index=False, engine="pyarrow")),
    benchmark("polars", lambda: df_pl.write_parquet(parquet_path))
)

io_results["Parquet read"] = (
    benchmark("pandas", lambda: pd.read_parquet(parquet_path, engine="pyarrow")),
    benchmark("polars", lambda: pl.read_parquet(parquet_path))
)

# --- PIVOTING ---
pivot_results = {}

# Narrower sample for pivot
pivot_pd = df_pd.copy()
pivot_pl = df_pl.clone()

pivot_results["Pivot wide (category→columns)"] = (
    benchmark("pandas", lambda: pivot_pd.pivot(index="timestamp", columns="category", values="value")),
    benchmark("polars", lambda: pivot_pl.pivot(index="timestamp", on="category", values="value", aggregate_function="first"))
)

pivot_wide_pd = pivot_pd.pivot(index="timestamp", columns="category", values="value")
pivot_wide_pl = pivot_pl.pivot(index="timestamp", on="category", values="value", aggregate_function="first")

pivot_results["Melt long (columns→category)"] = (
    benchmark("pandas", lambda: pivot_wide_pd.reset_index().melt(id_vars="timestamp", var_name="category", value_name="value")),
    benchmark("polars", lambda: pivot_wide_pl.unpivot(index="timestamp", on=pl.selectors.numeric(), value_name="value"))
)

# --- AFFICHAGE DES RÉSULTATS ---
print_results("Benchmarks sur les opérations TimeSeries", ts_results)
print_results("Benchmarks I/O CSV & Parquet", io_results)
print_results("Benchmarks Pivot Wide ↔ Long", pivot_results)



=== Benchmarks sur les opérations TimeSeries ===
Operation                           |     pandas |     polars | times faster (Polars)
--------------------------------------------------------------------------------
Filter (category == a)              |     0.1588 |     0.0071 | 22.26
Filter (timestamp between)          |     0.0213 |     0.0174 | 1.22
Mean per day                        |     0.3417 |     0.1014 | 3.37
Extract time features               |     0.1349 |     0.0574 | 2.35

=== Benchmarks I/O CSV & Parquet ===
Operation                           |     pandas |     polars | times faster (Polars)
--------------------------------------------------------------------------------
CSV write                           |     7.6342 |     0.5495 | 13.89
CSV read                            |     2.5590 |     0.0798 | 32.05
Parquet write                       |     0.4006 |     0.1882 | 2.13
Parquet read                        |     0.0692 |     0.0218 | 3.18

=== Benchmarks Pivot W

## Polars vs pandas – Key Differences

### Performance & Execution
- **Polars**: Multithreaded, lazy & eager modes
- **pandas**: Single-threaded, eager only

### Design Philosophy
- **Polars**: Immutable, columnar, Rust-based
- **pandas**: Mutable, row-based, Python/Cython, now arrow backend

### API & Syntax
- **Polars**: Expression-based (`pl.col`), no index, chainable (syntaxe comparable to `dplyr`).
- **pandas**: Index, comparable to `R` dataframe, no built to be chainable and seems to be incoherent (e.g bracket usages for index, methods)

### Time & Nulls
- **Polars**: Fast datetime, consistent nulls (`None`)
- **pandas**: `np.nan`, `pd.NA`, and `None` , `NaT` coexist

### I/O 
- **Polars**: Fast CSV/Parquet/JSON, limited ecosystem
- **pandas**: Slower I/O, rich ecosystem & library support
